<a href="https://colab.research.google.com/github/kshitij-nandanwar/GenAI/blob/main/GenAI_8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Step 1: Install libraries

In [ ]:
!pip install -q transformers accelerate "gradio>=5,<6"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.7/87.7 kB 925.8 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.5/63.5 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 325.6/325.6 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 462.4/462.4 kB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.3/74.3 kB 1.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-genai 2.12.1 requires pydantic<3.0.0,>=2.12.5, but you have pydantic 2.12.3 which is incompatible.
hf-gradio 0.4.1 requires gradio-client<3.0,>=2.0, but you have gradio-client 1.14.0 which is incompatible.
google-adk 2.7.1 requires starlette<2,>=1.3.1, but you have s

## Step 2: Load the LLM (Qwen2.5-1.5B-Instruct)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
).to(device)
print("Model loaded.")

Using device: cpu


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded.


## Step 3: Core generation function

In [ ]:
def generate(messages, temperature=0.7, max_new_tokens=500):
    """messages = list of {'role': 'system'|'user'|'assistant', 'content': str}"""
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=temperature > 0,
            temperature=max(temperature, 0.01),
            top_p=0.9,
            repetition_penalty=1.1,
        )
    new_tokens = output[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

# Quick test
print(generate([{"role": "user", "content": "Say hello in one sentence."}]))

Hello! How can I assist you today?


## Step 4: Prompt engineering: system prompts for each content type

In [ ]:
SYSTEM_PROMPTS = {
    "Email": """You are a professional business communication assistant.
Write clear, polite, well-structured emails.
Rules:
- Always include: Subject line, greeting, body (short paragraphs), closing, signature placeholder.
- Keep it under 200 words unless asked otherwise.
- Do not invent facts, names, dates or numbers; use [placeholders] instead.
Format:
Subject: ...
Dear ...,
...
Regards,
[Your Name]""",

    "Report": """You are a professional report writer.
Produce structured reports with these sections, using markdown headings:
1. Title
2. Executive Summary (2-3 sentences)
3. Background
4. Key Findings (bullet points)
5. Recommendations (numbered list)
6. Conclusion
Use formal language. If data is missing, say so instead of making up numbers.""",

    "Technical Explanation": """You are an expert technical educator.
Explain concepts step by step for a beginner.
Rules:
- Start with a one-line definition.
- Then give a simple real-world analogy.
- Then explain how it works in numbered steps.
- Add a short example (code if relevant).
- End with a 'Key Takeaways' list of 3 bullets.
Avoid unexplained jargon.""",
}

# Few-shot example: shows the model the exact style we want (Email mode)
FEW_SHOT = {
    "Email": [
        {"role": "user", "content": "Ask my manager for a day off next Friday for a family function."},
        {"role": "assistant", "content": """Subject: Leave Request for [Date]

Dear [Manager's Name],

I hope you are doing well. I would like to request leave on [Friday's date] to attend a family function.

I will complete my pending tasks beforehand and share a handover note so work continues smoothly in my absence. Please let me know if you need anything further from my side.

Thank you for your consideration.

Regards,
[Your Name]"""},
    ]
}

TONES = {
    "Formal": "Use a formal, professional tone.",
    "Friendly": "Use a warm, friendly but still professional tone.",
    "Persuasive": "Use a confident, persuasive tone.",
    "Concise": "Be as brief as possible while keeping the key points.",
}

def build_messages(mode, user_request, tone="Formal", history=None):
    system = SYSTEM_PROMPTS[mode] + "\n" + TONES[tone]
    messages = [{"role": "system", "content": system}]
    messages += FEW_SHOT.get(mode, [])          # few-shot examples
    if history:
        messages += history                      # multi-turn memory
    messages.append({"role": "user", "content": user_request})
    return messages

## Step 5: Compare a basic prompt vs an engineered prompt

In [ ]:
request = "Write an email to the team about the project deadline being extended by one week."

print("=== BASIC PROMPT (no engineering) ===")
print(generate([{"role": "user", "content": request}]))

print("\n=== ENGINEERED PROMPT (role + rules + format + few-shot) ===")
print(generate(build_messages("Email", request, tone="Formal")))

=== BASIC PROMPT (no engineering) ===
Subject: Project Deadline Extension Notice

Dear Team,

I hope this message finds you well.

As we all know, we have been working diligently on our latest project, and I am pleased to inform you that there has been a slight adjustment in the timeline for completion. Following discussions with our stakeholders and considering various factors, we have decided to extend the final deadline of the project by one week.

The new deadline is as follows:
- Original deadline: [Insert original date]
- New deadline: [Insert new date]

This extension will allow us to incorporate any additional feedback or considerations that may have arisen during the course of the project. It also gives us ample time to ensure that everything is completed accurately and within the scope agreed upon at the beginning of the project.

Please take this opportunity to review your individual responsibilities and workloads. Ensure that everyone is aware of their updated deadlines and

## Step 6: Test each content type

In [ ]:
print("##### REPORT #####")
print(generate(build_messages("Report", "Report on the benefits and risks of adopting remote work in a software company."), max_new_tokens=700))

print("\n##### TECHNICAL EXPLANATION #####")
print(generate(build_messages("Technical Explanation", "Explain what an API is."), max_new_tokens=600))

##### REPORT #####
## Title

Benefits and Risks of Adopting Remote Work in a Software Company

## Executive Summary

The adoption of remote work within a software company presents an opportunity to enhance productivity, flexibility, and team cohesion while mitigating certain challenges such as social isolation and communication barriers. This report outlines key findings from various studies and real-world implementations, highlighting both the advantages and potential drawbacks associated with remote work.

## Background

Remote work has gained significant traction globally due to its ability to reduce overhead costs for companies, increase employee autonomy, and promote work-life balance. The rise of remote work can also lead to improved collaboration among geographically dispersed teams, fostering innovation and creativity through diverse perspectives.

However, this shift towards remote work introduces several challenges. These include difficulties in maintaining effective communic

## Step 7: Effect of temperature

In [ ]:
req = "Write a short email inviting colleagues to a team lunch."
for t in [0.2, 1.0]:
    print(f"--- temperature = {t} ---")
    print(generate(build_messages("Email", req, "Friendly"), temperature=t, max_new_tokens=250))
    print()

--- temperature = 0.2 ---
Subject: Invitation to Team Lunch

Dear Colleagues,

We're excited to invite all of you to join us for a delicious lunch at our favorite restaurant this Thursday! It's an opportunity to catch up over some great food and conversation.

Please RSVP by [RSVP deadline]. We look forward to seeing everyone there!

Best regards,
[Your Name]

--- temperature = 1.0 ---
Subject: Team Lunch Invitation

Hi everyone,

Are we all ready to catch up after this long week?

Please come by for the usual Monday team lunch at noon on the designated spot.

See you there!

Best regards,

[Your Name]
[Your Title]



## Step 8: Build the chatbot UI (Gradio)

In [ ]:
import gradio as gr

def chat_fn(message, history, mode, tone, temperature):
    # history arrives as [{'role':..., 'content':...}, ...]
    past = [{"role": h["role"], "content": h["content"]} for h in history]
    msgs = build_messages(mode, message, tone, history=past)
    return generate(msgs, temperature=temperature, max_new_tokens=700)

demo = gr.ChatInterface(
    fn=chat_fn,
    type="messages",
    title="AI Content Generation Chatbot",
    description="Choose a content type and tone, then describe what you need. Follow up to refine (e.g. 'make it shorter').",
    additional_inputs=[
        gr.Dropdown(list(SYSTEM_PROMPTS.keys()), value="Email", label="Content type"),
        gr.Dropdown(list(TONES.keys()), value="Formal", label="Tone"),
        gr.Slider(0.0, 1.5, value=0.7, step=0.1, label="Temperature (creativity)"),
    ],
    examples=[
        ["Request a meeting with a client to discuss a project proposal.", "Email", "Formal", 0.7],
        ["Quarterly sales performance report for a retail store.", "Report", "Formal", 0.5],
        ["Explain how neural networks learn.", "Technical Explanation", "Friendly", 0.5],
    ],
)

demo.launch(share=True, debug=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f336d76698c525ea7c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
